# SoftLUT5 (CFGLUT5) Test — Single Reconfigurable LUT

This notebook validates that the Xilinx CFGLUT5 primitive works correctly on
the PYNQ-Z2. We program a single 5-input LUT with known truth tables and verify
all 32 input combinations produce the correct output.

## Prerequisites
Upload these files to `/home/xilinx/softlut5_test/` on the PYNQ board:
- `softlut5_test.bit` — bitstream from Vivado
- `softlut5_test.hwh` — hardware handoff (same base name!)

### AXI Address Map (axi_lut_ctrl.sv, 64KB)
| Offset | Description |
|--------|-------------|
| `0x0000` | Gate 0 truth table (write 32-bit INIT) |
| `0x8000` | STATUS (R) — bit 0 = cfg_busy |
| `0x8004` | TOTAL_GATES (R) — returns 1 |
| `0x9000` | NET_I input register (bits [4:0] = LUT inputs) |
| `0x9004` | NET_O output register (R) — bit 0 = LUT output |

## 1. Load the Overlay

In [ ]:
from pynq import Overlay, MMIO

BITSTREAM = "/home/xilinx/softlut5_test/softlut5_test.bit"
ol = Overlay(BITSTREAM)
print("Overlay loaded!")
print("IP dict keys:", list(ol.ip_dict.keys()))

In [ ]:
# Get MMIO handle
ip_name = [k for k in ol.ip_dict.keys() if "processing_system" not in k][0]
base_addr = ol.ip_dict[ip_name]['phys_addr']
print(f"Using IP: {ip_name}")
print(f"Base address: 0x{base_addr:08X}")

AXI_SPACE = 0x10000  # 64KB
mmio = MMIO(base_addr, AXI_SPACE)

## 2. AXI Constants & Helper Functions

In [ ]:
import time

ADDR_GATE_BASE  = 0x0000
ADDR_STATUS     = 0x8000
ADDR_GATE_COUNT = 0x8004
ADDR_INPUT      = 0x9000
ADDR_OUTPUT     = 0x9004  # NET_INPUTS=5, ceil(5/32)=1, so next word


def wait_ready(mmio, timeout_ms=100):
    """Wait for configuration shift FSM to finish."""
    start = time.time()
    while mmio.read(ADDR_STATUS) & 0x1:
        if (time.time() - start) * 1000 > timeout_ms:
            raise TimeoutError("cfg_busy timeout!")
        time.sleep(0.0001)


def program_gate(mmio, gate_id, init_value):
    """Program a gate's truth table (32-bit INIT)."""
    wait_ready(mmio)
    mmio.write(ADDR_GATE_BASE + gate_id * 4, init_value & 0xFFFFFFFF)
    wait_ready(mmio)  # Wait for the 32-cycle shift to finish


def run_lut(mmio, input_5bit):
    """Write 5-bit input, read 1-bit output."""
    mmio.write(ADDR_INPUT, input_5bit & 0x1F)
    return mmio.read(ADDR_OUTPUT) & 0x1

## 3. Smoke Test — Read Registers

In [ ]:
total_gates = mmio.read(ADDR_GATE_COUNT)
status = mmio.read(ADDR_STATUS)
print(f"TOTAL_GATES = {total_gates} (expected: 1)")
print(f"STATUS      = 0x{status:08X} (expected: 0 = idle)")
assert total_gates == 1, f"Expected 1 gate, got {total_gates}"
print("✅ Smoke test passed!")

## 4. Test 1: AND Gate

Program the LUT as a 5-input AND gate.

For LUT5, `INIT[addr]` where `addr = {I4,I3,I2,I1,I0}`.

5-input AND: output=1 only when all inputs=1, i.e., addr=31.
So `INIT = 0x80000000` (bit 31 set).

In [ ]:
# Program as AND gate
AND_INIT = 0x80000000
print(f"Programming gate 0 with INIT = 0x{AND_INIT:08X} (5-input AND)")
program_gate(mmio, 0, AND_INIT)
print("Programming done.")

# Sweep all 32 input combinations
errors = 0
for i in range(32):
    out = run_lut(mmio, i)
    expected = 1 if i == 31 else 0  # AND: only all-1s → 1
    if out != expected:
        print(f"  ❌ Input {i:05b} → got {out}, expected {expected}")
        errors += 1

if errors == 0:
    print(f"✅ AND gate: all 32 input combinations PASSED!")
else:
    print(f"❌ AND gate: {errors}/32 FAILED")

## 5. Test 2: XOR Gate (Parity)

Reprogram the same LUT as a 5-input XOR (parity) gate — output=1
when an odd number of inputs are 1.

`INIT = 0x96696996` (parity function for 5 inputs).

In [ ]:
# Program as 5-input parity (XOR)
PARITY_INIT = 0x96696996
print(f"Reprogramming gate 0 with INIT = 0x{PARITY_INIT:08X} (5-input parity)")
program_gate(mmio, 0, PARITY_INIT)
print("Reprogramming done — this proves runtime reconfigurability!")

# Sweep all 32 input combinations
errors = 0
for i in range(32):
    out = run_lut(mmio, i)
    expected = bin(i).count('1') % 2  # parity
    if out != expected:
        print(f"  ❌ Input {i:05b} → got {out}, expected {expected}")
        errors += 1

if errors == 0:
    print(f"✅ Parity gate: all 32 input combinations PASSED!")
    print(f"✅ Runtime reconfigurability VERIFIED!")
else:
    print(f"❌ Parity gate: {errors}/32 FAILED")

## 6. Test 3: 2-input AND (LUT2 mode)

Our actual LLNN model uses LUT2 (2-input) neurons with I4=I3=I2=0.
For a 2-input AND: output=1 only when I0=I1=1.

With I4=I3=I2=0, the address is just `{0,0,0,I1,I0}`, so:
- addr=0 (00) → 0
- addr=1 (01) → 0
- addr=2 (10) → 0
- addr=3 (11) → 1

`INIT[3]=1`, all others 0 → `INIT = 0x00000008`

In [ ]:
# LUT2 AND gate: INIT = 0x00000008 (bit 3 = {0,0,0,1,1})
LUT2_AND_INIT = 0x00000008
print(f"Programming gate 0 with INIT = 0x{LUT2_AND_INIT:08X} (2-input AND, LUT2 mode)")
program_gate(mmio, 0, LUT2_AND_INIT)

# Test all 4 combinations of I0/I1 (with I2=I3=I4=0)
errors = 0
for i1 in range(2):
    for i0 in range(2):
        inp = (i1 << 1) | i0
        out = run_lut(mmio, inp)  # only bits [1:0] vary, rest are 0
        expected = i0 & i1
        status = "✅" if out == expected else "❌"
        print(f"  I1={i1} I0={i0} → out={out} (expected {expected}) {status}")
        if out != expected:
            errors += 1

if errors == 0:
    print(f"\n✅ LUT2 AND gate: all 4 combinations PASSED!")
    print(f"   This matches how our LLNN model neurons are configured.")
else:
    print(f"\n❌ LUT2 AND gate: {errors}/4 FAILED")

## 7. Summary

If all tests passed:
1. ✅ CFGLUT5 primitive works on the 7020
2. ✅ AXI serial shift FSM programs it correctly (MSB-first, 32 cycles)
3. ✅ Runtime reconfigurability is verified (AND → Parity → LUT2 AND)

**Next step:** Scale up to the full reconfigurable overlay with 1512+ SoftLUT5 instances.

In [ ]:
print("="*50)
print("SoftLUT5 (CFGLUT5) Test Complete")
print("="*50)
print()
print("If all tests above show ✅, the CFGLUT5 primitive")
print("is working correctly and we can proceed to build")
print("the full reconfigurable LLNN overlay.")